# OCR DateCode — SupCon 128-class (char × ok/ng)

EfficientNet-B0 + Projection head trained with **Supervised Contrastive Loss**.

**Idea**: thay vì binary OK/NG, gán label `(char_X, ok)` và `(char_X, ng)` → tới 128 class.
SupCon kéo các sample cùng class lại gần, đẩy khác class ra xa trong vector space (cosine).
Inference: embed → cosine sim với 128 class centroids → nearest → OK hoặc NG.

**Pipeline**
1. Setup
2. Config
3. Dataset 128-class
4. Augment + TwoView (cần view diversity cho contrastive)
5. Visualize data
6. Model (backbone + projection 128-d L2-norm)
7. Train SupCon
8. Compute centroids
9. Evaluation (nearest-centroid + threshold sweep)
10. **UMAP embedding plot** — xem OK/NG có tách thật không
11. NG analysis
12. ONNX export


## 1. Setup


In [ ]:
!pip install -q timm albumentations onnx onnxruntime scikit-learn matplotlib pandas \
    pytorch-metric-learning umap-learn


In [ ]:
# Optional: mount Drive
# from google.colab import drive
# drive.mount('/content/drive')


## 2. Config


In [ ]:
# === EDIT ME ===
DATASET_ZIP   = '/content/dataset.zip'
DATASET_ROOT  = '/content/dataset'
OUTPUT_DIR    = '/content/runs/supcon_128'

IMAGE_SIZE     = 64
EPOCHS         = 50          # SupCon thường cần nhiều epoch hơn CE
BATCH_SIZE     = 128         # mỗi batch sẽ x2 do TwoView → effective 256
LR             = 3e-4
WEIGHT_DECAY   = 1e-4
NUM_WORKERS    = 2
VAL_RATIO      = 0.15
MIN_PER_CLASS  = 4           # bỏ class có < 4 mẫu (tránh SupCon bất ổn)
SUPCON_TEMP    = 0.07
PROJ_DIM       = 128
SEED           = 42

import os, random, json, math
import numpy as np, torch
from pathlib import Path
os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# Extract dataset
if not Path(DATASET_ROOT).exists() or not any(Path(DATASET_ROOT).glob('char_*')):
    !mkdir -p {DATASET_ROOT}
    !unzip -q -o {DATASET_ZIP} -d {DATASET_ROOT}
    inner = list(Path(DATASET_ROOT).iterdir())
    if len(inner) == 1 and inner[0].is_dir() and not any(Path(DATASET_ROOT).glob('char_*')):
        nested = inner[0]
        for p in nested.iterdir():
            p.rename(Path(DATASET_ROOT) / p.name)
        nested.rmdir()
print('Char folders:', len(list(Path(DATASET_ROOT).glob('char_*'))))


## 3. Dataset 128-class

Label = `f"{char_folder}_{ok|ng}"`. Class index dạng int. Bỏ class < `MIN_PER_CLASS` mẫu.


In [ ]:
import cv2
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader

VALID_EXT = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

def scan_dataset(root, min_per_class=4):
    """Return (samples, class_to_idx) where samples = [(path, class_idx, ok_or_ng_int, char_folder), ...]."""
    raw = defaultdict(list)
    for char_dir in sorted(Path(root).iterdir()):
        if not char_dir.is_dir() or not char_dir.name.startswith('char_'):
            continue
        for sub in ['ok', 'ng']:
            folder = char_dir / sub
            if not folder.is_dir():
                continue
            cls_name = f'{char_dir.name}__{sub}'
            for p in sorted(folder.iterdir()):
                if p.suffix.lower() in VALID_EXT:
                    raw[cls_name].append((str(p), char_dir.name, sub))
    # Filter classes with too few samples
    raw = {k: v for k, v in raw.items() if len(v) >= min_per_class}
    class_to_idx = {k: i for i, k in enumerate(sorted(raw.keys()))}
    samples = []
    for cls_name, items in raw.items():
        idx = class_to_idx[cls_name]
        for path, char, sub in items:
            samples.append((path, idx, 0 if sub == 'ok' else 1, char))
    return samples, class_to_idx


class CharSupConDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, ok_ng, char = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is None:
            from PIL import Image
            img = cv2.cvtColor(np.array(Image.open(path).convert('RGB')), cv2.COLOR_RGB2BGR)
        if self.transform is not None:
            img = self.transform(image=img)['image']
        return img, label, ok_ng, char, path


def stratified_split(samples, val_ratio=0.15, seed=42):
    rng = random.Random(seed)
    by_cls = defaultdict(list)
    for s in samples:
        by_cls[s[1]].append(s)
    train, val = [], []
    for cls, items in by_cls.items():
        items = list(items); rng.shuffle(items)
        n_val = max(1, int(round(len(items) * val_ratio)))
        if n_val >= len(items):
            n_val = max(1, len(items) - 1)
        val.extend(items[:n_val])
        train.extend(items[n_val:])
    return train, val


all_samples, class_to_idx = scan_dataset(DATASET_ROOT, MIN_PER_CLASS)
idx_to_class = {v: k for k, v in class_to_idx.items()}
train_samples, val_samples = stratified_split(all_samples, VAL_RATIO, SEED)
print(f'Total samples: {len(all_samples)}  classes: {len(class_to_idx)}')
print(f'Train: {len(train_samples)}  Val: {len(val_samples)}')

# Map class_idx → ok/ng int (for fast lookup at eval)
class_idx_to_ok_ng = {idx: (0 if name.endswith('__ok') else 1)
                      for name, idx in class_to_idx.items()}


## 4. Augmentation (medium) + TwoView

Contrastive cần view diversity → mạnh hơn binary CE notebook một chút, nhưng vẫn tôn trọng synth.
Hai view augment khác nhau cho cùng 1 ảnh → SupCon coi 2 view là positive pair.


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_train_tf(size):
    return A.Compose([
        A.Affine(translate_percent=(-0.05, 0.05), rotate=(-3, 3), scale=(0.92, 1.08),
                 interpolation=1, cval=255, p=0.7),
        A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), sigma_limit=(0.3, 0.9), p=1.0),
            A.GaussNoise(var_limit=(5.0, 20.0), p=1.0),
        ], p=0.25),
        A.CoarseDropout(max_holes=2, max_height=5, max_width=5, fill_value=255, p=0.10),
        A.LongestMaxSize(max_size=size, interpolation=1),
        A.PadIfNeeded(min_height=size, min_width=size, border_mode=0, value=(255,255,255)),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])

def build_eval_tf(size):
    return A.Compose([
        A.LongestMaxSize(max_size=size, interpolation=1),
        A.PadIfNeeded(min_height=size, min_width=size, border_mode=0, value=(255,255,255)),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, max_pixel_value=255.0),
        ToTensorV2(),
    ])

train_tf = build_train_tf(IMAGE_SIZE)
val_tf   = build_eval_tf(IMAGE_SIZE)


class TwoViewDataset(Dataset):
    """Wraps a CharSupConDataset to return two independently-augmented views."""
    def __init__(self, base):
        self.base = base
    def __len__(self):
        return len(self.base)
    def __getitem__(self, idx):
        v1, label, ok_ng, char, path = self.base[idx]
        v2, _, _, _, _ = self.base[idx]
        return v1, v2, label, ok_ng, char, path

print('Augment + TwoView ready.')


## 5. Visualize data (128-class distribution)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Per-class count
per_cls = defaultdict(int)
for _, lbl, _, _ in all_samples:
    per_cls[lbl] += 1

# Group by char and ok/ng for plot
char_ok = defaultdict(int); char_ng = defaultdict(int)
for _, lbl, ok_ng, char in all_samples:
    if ok_ng == 0: char_ok[char] += 1
    else:          char_ng[char] += 1

chars = sorted(set(list(char_ok.keys()) + list(char_ng.keys())))
ok_v  = [char_ok.get(c, 0) for c in chars]
ng_v  = [char_ng.get(c, 0) for c in chars]
print(f'Active classes: {len(per_cls)}  (filtered min={MIN_PER_CLASS})')
print(f'Total OK: {sum(ok_v)}  Total NG: {sum(ng_v)}')

fig, ax = plt.subplots(figsize=(18, 4))
x = np.arange(len(chars))
ax.bar(x, ok_v, label='OK', color='steelblue')
ax.bar(x, ng_v, bottom=ok_v, label='NG', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(chars, rotation=90, fontsize=7)
ax.set_title('Samples per char (OK + NG stacked)'); ax.legend()
plt.tight_layout(); plt.show()

# Class-level histogram
counts = sorted(per_cls.values())
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(counts, marker='.')
ax.set_title(f'Sample-count distribution across {len(per_cls)} classes (sorted)')
ax.set_xlabel('class rank'); ax.set_ylabel('count'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
# Sample grids
def show_grid(samples_subset, title, n=24, cols=8):
    if not samples_subset:
        print(f'{title}: empty'); return
    rng = np.random.default_rng(0)
    pick = rng.choice(len(samples_subset), size=min(n, len(samples_subset)), replace=False)
    rows_n = (len(pick) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*1.6, rows_n*1.7))
    axes = np.atleast_2d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(pick):
            p, lbl, ok_ng, c = samples_subset[pick[i]][:4]
            img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(f'{c}/{"OK" if ok_ng==0 else "NG"}', fontsize=7)
    plt.suptitle(title, fontsize=11); plt.tight_layout(); plt.show()

ok_pool = [s for s in all_samples if s[2] == 0]
ng_pool = [s for s in all_samples if s[2] == 1]
show_grid(ok_pool, 'Random OK samples', n=24)
show_grid(ng_pool, 'Random NG samples', n=24)


## 6. Model — Backbone + Projection head

EfficientNet-B0 → 1280-d → MLP(512) → 128-d L2-normalized.
Output dùng cho SupCon (cosine geometry).


In [ ]:
import timm
import torch.nn as nn
import torch.nn.functional as F

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, hidden=512, out_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.bn  = nn.BatchNorm1d(hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = F.relu(x, inplace=True)
        x = self.fc2(x)
        return F.normalize(x, dim=-1)


class ContrastiveModel(nn.Module):
    def __init__(self, backbone='efficientnet_b0', proj_dim=128, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone, pretrained=pretrained,
                                          num_classes=0, global_pool='avg')
        feat_dim = self.backbone.num_features
        self.projector = ProjectionHead(feat_dim, hidden=512, out_dim=proj_dim)
        self.feat_dim = feat_dim
        self.proj_dim = proj_dim

    def forward(self, x):
        f = self.backbone(x)
        z = self.projector(f)
        return z   # L2-normalized 128-d

model = ContrastiveModel(proj_dim=PROJ_DIM).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'ContrastiveModel: efficientnet_b0 + proj_head  params={n_params:.2f}M')
with torch.no_grad():
    z = model(torch.randn(2, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE))
print('Forward OK. embedding shape:', z.shape, 'norm:', z.norm(dim=1)[:2].cpu().numpy())


## 7. Train SupCon


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from pytorch_metric_learning.losses import SupConLoss

train_base = CharSupConDataset(train_samples, transform=train_tf)
val_base   = CharSupConDataset(val_samples,   transform=val_tf)
train_ds   = TwoViewDataset(train_base)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_base, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

# Reset model (clean slate for training cell re-runs)
model = ContrastiveModel(proj_dim=PROJ_DIM).to(DEVICE)
opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched = CosineAnnealingLR(opt, T_max=EPOCHS)
sup_con_loss = SupConLoss(temperature=SUPCON_TEMP)

print(f'Train batches/epoch: {len(train_loader)}  effective batch (with TwoView): {BATCH_SIZE*2}')


In [ ]:
@torch.no_grad()
def compute_centroids(model, loader, num_classes, device):
    model.eval()
    sums = torch.zeros(num_classes, model.proj_dim, device=device)
    counts = torch.zeros(num_classes, device=device)
    for batch in loader:
        if len(batch) == 6:    # TwoView
            v1, _, lbl, _, _, _ = batch
        else:                   # base
            v1, lbl, _, _, _ = batch
        v1 = v1.to(device, non_blocking=True)
        lbl = lbl.to(device)
        z = model(v1)
        for c in lbl.unique():
            mask = (lbl == c)
            sums[c] += z[mask].sum(dim=0)
            counts[c] += mask.sum()
    centroids = sums / counts.clamp(min=1).unsqueeze(1)
    centroids = F.normalize(centroids, dim=1)
    return centroids


@torch.no_grad()
def quick_val_eval(model, val_loader, centroids, device):
    """Fast eval: nearest-centroid acc + binary OK/NG acc on val."""
    model.eval()
    correct_cls = 0; correct_bin = 0; total = 0
    for v, lbl, ok_ng, _, _ in val_loader:
        v = v.to(device); lbl = lbl.to(device); ok_ng = ok_ng.to(device)
        z = model(v)
        sims = z @ centroids.T
        pred = sims.argmax(dim=1)
        # Binary: predicted class' ok_ng tag
        pred_ok_ng = torch.tensor(
            [class_idx_to_ok_ng[int(p)] for p in pred.cpu().tolist()],
            device=device,
        )
        correct_cls += (pred == lbl).sum().item()
        correct_bin += (pred_ok_ng == ok_ng).sum().item()
        total += lbl.size(0)
    return correct_cls / total, correct_bin / total


# Build a single-view loader for centroid computation
centroid_loader = DataLoader(
    CharSupConDataset(train_samples, transform=val_tf),
    batch_size=BATCH_SIZE*2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print('Helpers ready.')


In [ ]:
best_score = -1.0
best_path = os.path.join(OUTPUT_DIR, 'best.pt')
history = {'loss': [], 'cls_acc': [], 'bin_acc': []}

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0; n = 0
    for v1, v2, lbl, _, _, _ in train_loader:
        v1 = v1.to(DEVICE, non_blocking=True)
        v2 = v2.to(DEVICE, non_blocking=True)
        lbl = lbl.to(DEVICE, non_blocking=True)
        embs = torch.cat([model(v1), model(v2)], dim=0)
        labels = torch.cat([lbl, lbl], dim=0)
        loss = sup_con_loss(embs, labels)
        opt.zero_grad(); loss.backward(); opt.step()
        bs = v1.size(0)
        epoch_loss += loss.item() * bs; n += bs
    epoch_loss /= max(1, n)
    sched.step()

    centroids = compute_centroids(model, centroid_loader,
                                   num_classes=len(class_to_idx), device=DEVICE)
    cls_acc, bin_acc = quick_val_eval(model, val_loader, centroids, DEVICE)
    score = bin_acc   # main objective is OK/NG accuracy
    history['loss'].append(epoch_loss)
    history['cls_acc'].append(cls_acc)
    history['bin_acc'].append(bin_acc)

    flag = ''
    if score > best_score:
        best_score = score
        torch.save({
            'model': model.state_dict(),
            'centroids': centroids.cpu(),
            'class_to_idx': class_to_idx,
            'epoch': epoch,
            'score': score,
        }, best_path)
        flag = ' ← saved best'
    print(f'Ep {epoch+1:02d}/{EPOCHS}  supcon_loss={epoch_loss:.4f}  '
          f'cls_acc={cls_acc:.3f}  bin_acc(OK/NG)={bin_acc:.3f}{flag}')

print(f'\nBest binary OK/NG acc: {best_score:.4f}')


In [ ]:
# Plot history
fig, axes = plt.subplots(1, 2, figsize=(13, 3.2))
axes[0].plot(history['loss'], color='steelblue'); axes[0].set_title('SupCon loss'); axes[0].grid(alpha=0.3)
axes[1].plot(history['cls_acc'], label='128-class acc',  color='gray')
axes[1].plot(history['bin_acc'], label='binary OK/NG',    color='tomato')
axes[1].set_title('Val accuracy'); axes[1].set_ylim(0, 1.02); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 8. Evaluation (load best, full inference)

- Compute embeddings cho cả val
- Cosine sim với 128 centroids → nearest class → OK/NG
- Threshold sweep trên `max_sim` để dùng làm anomaly fallback (optional)


In [ ]:
ckpt = torch.load(best_path, map_location=DEVICE)
model.load_state_dict(ckpt['model'])
centroids = ckpt['centroids'].to(DEVICE)
print(f'Loaded best (epoch {ckpt["epoch"]+1}, score {ckpt["score"]:.4f})')

# Full val inference
all_emb, all_lbl, all_ok_ng, all_char, all_path = [], [], [], [], []
model.eval()
with torch.no_grad():
    for v, lbl, ok_ng, char, path in val_loader:
        v = v.to(DEVICE)
        z = model(v).cpu().numpy()
        all_emb.append(z)
        all_lbl.extend(lbl.numpy().tolist())
        all_ok_ng.extend(ok_ng.numpy().tolist())
        all_char.extend(list(char))
        all_path.extend(list(path))

all_emb = np.concatenate(all_emb, axis=0)        # (N, 128)
all_lbl = np.array(all_lbl)
all_ok_ng = np.array(all_ok_ng)
print('Embeddings:', all_emb.shape)

# Cosine sim with centroids
cents_np = centroids.cpu().numpy()
sims = all_emb @ cents_np.T                      # (N, num_class)
pred_cls = sims.argmax(axis=1)
max_sim  = sims.max(axis=1)

pred_ok_ng = np.array([class_idx_to_ok_ng[int(p)] for p in pred_cls])
cls_acc = (pred_cls == all_lbl).mean()
bin_acc = (pred_ok_ng == all_ok_ng).mean()
print(f'128-class acc: {cls_acc:.4f}')
print(f'Binary OK/NG acc: {bin_acc:.4f}')

# Sim score for ROC: use NG-class cosine sim
# For each sample, prob_ng-like score = max sim to NG centroids - max sim to OK centroids
ng_mask = np.array([class_idx_to_ok_ng[i] == 1 for i in range(len(class_to_idx))])
ok_mask = ~ng_mask
sim_to_ng = sims[:, ng_mask].max(axis=1) if ng_mask.any() else np.full(len(sims), -1.0)
sim_to_ok = sims[:, ok_mask].max(axis=1) if ok_mask.any() else np.full(len(sims), -1.0)
score_ng = sim_to_ng - sim_to_ok    # >0 → NG more likely


In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix

auc = roc_auc_score(all_ok_ng, score_ng)
print(f'ROC-AUC (sim_ng - sim_ok): {auc:.4f}')

# Threshold sweep (default decision = nearest centroid; threshold sweep is optional fine-tune)
ths = np.linspace(score_ng.min(), score_ng.max(), 91)
best_th, best_balanced = 0.0, -1.0
sweep_rows = []
for th in ths:
    pred = (score_ng >= th).astype(int)
    tp = int(((pred==1) & (all_ok_ng==1)).sum())
    fp = int(((pred==1) & (all_ok_ng==0)).sum())
    tn = int(((pred==0) & (all_ok_ng==0)).sum())
    fn = int(((pred==0) & (all_ok_ng==1)).sum())
    okp = tn / max(1, tn+fp); ngc = tp / max(1, tp+fn)
    sc = 0.5*okp + 0.5*ngc
    sweep_rows.append({'th': th, 'ok_pass': okp, 'ng_catch': ngc, 'balanced': sc})
    if sc > best_balanced:
        best_balanced = sc; best_th = th
sweep_df = pd.DataFrame(sweep_rows)
print(f'Best score-threshold: {best_th:.4f}  →  balanced={best_balanced:.4f}')

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(sweep_df.th, sweep_df.ok_pass,  label='OK pass',  color='steelblue')
ax.plot(sweep_df.th, sweep_df.ng_catch, label='NG catch', color='tomato')
ax.plot(sweep_df.th, sweep_df.balanced, label='balanced', color='black', linestyle='--')
ax.axvline(best_th, color='green', linestyle=':', label=f'best={best_th:.3f}')
ax.set_xlabel('threshold (sim_ng - sim_ok)'); ax.set_ylabel('rate')
ax.set_ylim(0, 1.02); ax.grid(alpha=0.3); ax.legend()
ax.set_title('Threshold sweep on (sim_ng - sim_ok)')
plt.tight_layout(); plt.show()


In [ ]:
# Final report — use score-threshold (often slightly better than nearest-centroid)
preds_th = (score_ng >= best_th).astype(int)
print('=== Threshold-based OK/NG ===')
print(classification_report(all_ok_ng, preds_th, target_names=['OK','NG'], digits=4))

print('=== Nearest-centroid OK/NG ===')
print(classification_report(all_ok_ng, pred_ok_ng, target_names=['OK','NG'], digits=4))

cm = confusion_matrix(all_ok_ng, preds_th)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]; ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(['Pred OK','Pred NG'])
ax.set_yticks([0,1]); ax.set_yticklabels(['True OK','True NG'])
ax.set_title(f'Confusion (th={best_th:.3f})')
fpr, tpr, _ = roc_curve(all_ok_ng, score_ng)
ax = axes[1]; ax.plot(fpr, tpr, color='tomato', label=f'AUC={auc:.4f}')
ax.plot([0,1],[0,1],'k--',alpha=0.4); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 9. Embedding visualization (UMAP)

Vẽ embedding 128-d xuống 2D để xem **OK/NG có thật sự tách trong không gian không**.
Đây là điểm mạnh nhất của SupCon vs CE.


In [ ]:
import umap

reducer = umap.UMAP(n_neighbors=20, min_dist=0.1, metric='cosine', random_state=SEED)
emb_2d = reducer.fit_transform(all_emb)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# (a) Color by ok/ng
ax = axes[0]
ok_idx = (all_ok_ng == 0); ng_idx = (all_ok_ng == 1)
ax.scatter(emb_2d[ok_idx, 0], emb_2d[ok_idx, 1], s=4, alpha=0.5, color='steelblue', label=f'OK ({ok_idx.sum()})')
ax.scatter(emb_2d[ng_idx, 0], emb_2d[ng_idx, 1], s=4, alpha=0.5, color='tomato',    label=f'NG ({ng_idx.sum()})')
ax.set_title('UMAP — colored by OK/NG'); ax.legend(); ax.set_xticks([]); ax.set_yticks([])

# (b) Color by char (categorical)
ax = axes[1]
char_set = sorted(set(all_char))
cmap = plt.cm.get_cmap('tab20', max(20, len(char_set)))
char_to_color = {c: cmap(i % cmap.N) for i, c in enumerate(char_set)}
colors = [char_to_color[c] for c in all_char]
ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=4, alpha=0.6, c=colors)
ax.set_title(f'UMAP — colored by char ({len(char_set)} chars)'); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 10. NG analysis

Dùng **threshold-based prediction** (preds_th) cho NG analysis vì nó là decision cuối cùng để deploy.


In [ ]:
df = pd.DataFrame({
    'path': all_path,
    'char': all_char,
    'true_label_idx': all_lbl,
    'true_class_name': [idx_to_class[l] for l in all_lbl],
    'pred_class_idx': pred_cls,
    'pred_class_name': [idx_to_class[p] for p in pred_cls],
    'true_ok_ng': all_ok_ng,
    'pred_ok_ng_centroid': pred_ok_ng,
    'pred_ok_ng_th': preds_th,
    'max_sim': max_sim,
    'sim_to_ok': sim_to_ok,
    'sim_to_ng': sim_to_ng,
    'score_ng': score_ng,
})
df['kind'] = df.apply(
    lambda r: ('TN','FN','FP','TP')[r['true_ok_ng']*2 + r['pred_ok_ng_th']],
    axis=1
)
print(df['kind'].value_counts().to_string())


In [ ]:
# Per-char breakdown
rows = []
for ch, g in df.groupby('char'):
    ok = g[g['true_ok_ng']==0]; ng = g[g['true_ok_ng']==1]
    rows.append({
        'char': ch,
        'n_ok': len(ok), 'n_ng': len(ng),
        'ok_pass':  (ok['pred_ok_ng_th']==0).mean()  if len(ok)>0 else float('nan'),
        'ng_catch': (ng['pred_ok_ng_th']==1).mean()  if len(ng)>0 else float('nan'),
        'fn_count': int((ng['pred_ok_ng_th']==0).sum()) if len(ng)>0 else 0,
        'fp_count': int((ok['pred_ok_ng_th']==1).sum()) if len(ok)>0 else 0,
    })
per_char_df = pd.DataFrame(rows)

print('=== 15 chars worst NG catch ===')
print(per_char_df.dropna(subset=['ng_catch']).sort_values('ng_catch').head(15).to_string(index=False))
print()
print('=== 15 chars worst OK pass ===')
print(per_char_df.dropna(subset=['ok_pass']).sort_values('ok_pass').head(15).to_string(index=False))

plot_df = per_char_df.sort_values('char')
fig, ax = plt.subplots(figsize=(18, 4))
x = np.arange(len(plot_df)); w = 0.4
ax.bar(x - w/2, plot_df.ok_pass.fillna(0),  width=w, label='OK pass',  color='steelblue')
ax.bar(x + w/2, plot_df.ng_catch.fillna(0), width=w, label='NG catch', color='tomato')
ax.set_xticks(x); ax.set_xticklabels(plot_df.char, rotation=90, fontsize=7)
ax.set_ylim(0, 1.02); ax.legend(); ax.set_title('Per-char OK pass vs NG catch')
ax.axhline(0.9, color='gray', linestyle='--', linewidth=0.5)
plt.tight_layout(); plt.show()


In [ ]:
def gallery(sub_df, title, n=24, cols=8, sort_by=None, ascending=True):
    sub = sub_df.copy()
    if sort_by is not None:
        sub = sub.sort_values(sort_by, ascending=ascending)
    sub = sub.head(n)
    if len(sub) == 0:
        print(f'{title}: empty'); return
    rows_n = (len(sub) + cols - 1) // cols
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols*1.7, rows_n*1.95))
    axes = np.atleast_2d(axes).ravel()
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(sub):
            r = sub.iloc[i]
            img = cv2.cvtColor(cv2.imread(r['path']), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            tag = (f"{r['char']}\n"
                   f"P_NG={r['score_ng']:.2f}\n"
                   f"pred:{r['pred_class_name'].split('__')[-1]}")
            color = 'red' if r['kind'] in ('FN','FP') else 'black'
            ax.set_title(tag, fontsize=7, color=color)
    plt.suptitle(f'{title}  (n={len(sub)})', fontsize=11)
    plt.tight_layout(); plt.show()


fn_df = df[df['kind'] == 'FN']
fp_df = df[df['kind'] == 'FP']
tp_df = df[df['kind'] == 'TP'].copy(); tp_df['margin'] = tp_df['score_ng'] - best_th

print(f'False Negatives: {len(fn_df)}')
gallery(fn_df, 'False Negatives — NG escaped as OK', sort_by='score_ng', ascending=False)

print(f'False Positives: {len(fp_df)}')
gallery(fp_df, 'False Positives — OK rejected as NG', sort_by='score_ng', ascending=True)

gallery(tp_df, 'Hardest correctly-classified NG (low margin)', sort_by='margin', ascending=True)


In [ ]:
# Save analysis CSVs
fn_path = os.path.join(OUTPUT_DIR, 'false_negatives.csv')
fp_path = os.path.join(OUTPUT_DIR, 'false_positives.csv')
per_char_path = os.path.join(OUTPUT_DIR, 'per_char_metrics.csv')

fn_df[['path','char','true_class_name','pred_class_name','score_ng']].to_csv(fn_path, index=False)
fp_df[['path','char','true_class_name','pred_class_name','score_ng']].to_csv(fp_path, index=False)
per_char_df.to_csv(per_char_path, index=False)
print('Saved CSVs.')


## 11. ONNX export + artifacts

Encoder ONNX (input → 128-d L2-norm embedding) + centroids.npy + class_names.json + meta.json.

Inference pipeline:
```
embedding = onnx_session.run(input)[0]
sims = embedding @ centroids.T                # (1, num_class)
pred_class = class_names[sims.argmax()]
is_ng = pred_class.endswith('__ng')
```


In [ ]:
# Export encoder ONNX (single-file, no .data sidecar)
import onnx
model.eval()
onnx_path = os.path.join(OUTPUT_DIR, 'supcon_encoder.onnx')
dummy = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)
torch.onnx.export(
    model, dummy, onnx_path,
    input_names=['input'], output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17, do_constant_folding=True,
)

# Force single-file
m = onnx.load(onnx_path, load_external_data=True)
onnx.save_model(m, onnx_path, save_as_external_data=False)
for sidecar in Path(OUTPUT_DIR).glob('supcon_encoder*.data'):
    sidecar.unlink()
for sidecar in Path(OUTPUT_DIR).glob('supcon_encoder.onnx.data'):
    sidecar.unlink()

size_mb = os.path.getsize(onnx_path) / 1e6
print(f'Exported {onnx_path}  ({size_mb:.2f} MB, single-file)')

# Verify
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
test_in = np.random.randn(32, 3, IMAGE_SIZE, IMAGE_SIZE).astype(np.float32)
out = sess.run(None, {'input': test_in})[0]
print('ONNX embedding shape:', out.shape)
print('Norms (first 4):', np.linalg.norm(out, axis=1)[:4])


In [ ]:
# Save centroids + meta
centroids_path  = os.path.join(OUTPUT_DIR, 'centroids.npy')
classnames_path = os.path.join(OUTPUT_DIR, 'class_names.json')
meta_path       = os.path.join(OUTPUT_DIR, 'model_meta.json')

np.save(centroids_path, cents_np)

# class_names: list ordered by class_idx, each entry: {'name', 'char', 'kind' (ok/ng)}
class_names = []
for idx in range(len(class_to_idx)):
    name = idx_to_class[idx]
    char_part, kind = name.rsplit('__', 1)
    class_names.append({'idx': idx, 'name': name, 'char': char_part, 'kind': kind})
with open(classnames_path, 'w') as f:
    json.dump(class_names, f, indent=2)

meta = {
    'image_size': IMAGE_SIZE,
    'embed_dim': PROJ_DIM,
    'num_classes': len(class_to_idx),
    'normalization': {'mean': list(IMAGENET_MEAN), 'std': list(IMAGENET_STD)},
    'inference_method': 'nearest_centroid + threshold',
    'threshold_score_ng': float(best_th),
    'threshold_explanation': 'score = max_sim_to_ng_centroid - max_sim_to_ok_centroid; if score >= threshold → NG',
    'metrics': {
        'auc': float(auc),
        'bin_accuracy_centroid': float(bin_acc),
        'bin_balanced_threshold': float(best_balanced),
        'cls_accuracy_128': float(cls_acc),
    },
    'training': {
        'backbone': 'efficientnet_b0',
        'loss': 'SupCon',
        'temperature': SUPCON_TEMP,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
    },
}
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved:', centroids_path, classnames_path, meta_path)
print(json.dumps(meta, indent=2))


In [ ]:
# Download all artifacts
from google.colab import files
files.download(onnx_path)
files.download(centroids_path)
files.download(classnames_path)
files.download(meta_path)
files.download(per_char_path)
files.download(fn_path)
files.download(fp_path)


## 12. Inference snippet (paste into desktop / production)

```python
import cv2, numpy as np, json, onnxruntime as ort

sess = ort.InferenceSession('supcon_encoder.onnx', providers=['CPUExecutionProvider'])
centroids = np.load('centroids.npy')                             # (C, 128)
class_names = json.load(open('class_names.json'))                # list of {idx, name, char, kind}
meta = json.load(open('model_meta.json'))
SIZE = meta['image_size']; TH = meta['threshold_score_ng']
MEAN = np.array(meta['normalization']['mean'], dtype=np.float32)
STD  = np.array(meta['normalization']['std'],  dtype=np.float32)

ng_mask = np.array([cn['kind'] == 'ng' for cn in class_names])
ok_mask = ~ng_mask

def preprocess(bgr):
    h, w = bgr.shape[:2]
    s = SIZE / max(h, w)
    nh, nw = int(round(h*s)), int(round(w*s))
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_LINEAR)
    canvas = np.full((SIZE, SIZE, 3), 255, dtype=np.uint8)
    y0, x0 = (SIZE-nh)//2, (SIZE-nw)//2
    canvas[y0:y0+nh, x0:x0+nw] = img
    rgb = cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - MEAN) / STD
    return rgb.transpose(2, 0, 1)[None]

def predict(bgr):
    x = preprocess(bgr).astype(np.float32)
    emb = sess.run(None, {'input': x})[0][0]                      # (128,)
    sims = emb @ centroids.T                                      # (C,)
    score_ng = sims[ng_mask].max() - sims[ok_mask].max()
    pred_idx = sims.argmax()
    pred_char = class_names[pred_idx]['char']
    is_ng = score_ng >= TH
    return {'is_ng': bool(is_ng), 'score_ng': float(score_ng),
            'pred_char_folder': pred_char, 'pred_class': class_names[pred_idx]['name']}
```
